In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import wandb
import random
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
wandb.init(
    project="MNIST",
    config={
        "epochs": 20,
        "batch_size": 128,
        "lr": 0.01,
        "architecture": "cnn"
    }
)
config = wandb.config

In [ ]:
train_transform = transforms.Compose([
transforms.RandomRotation(10),
transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
transforms.ToTensor(),
])
test_transform = transforms.Compose([
transforms.ToTensor(),
])
train_dataset = datasets.MNIST(
root="./data",
train=True,
download=True,
transform=train_transform
)
test_dataset = datasets.MNIST(
root="./data",
train=False,
download=True,
transform=test_transform
)
train_loader = DataLoader(
train_dataset,
batch_size=config.batch_size,
shuffle=True
)
test_loader = DataLoader(
test_dataset,
batch_size=config.batch_size,
shuffle=False
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 137MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 17.0MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 89.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.13MB/s]


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.03)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(32,64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.05)
        )

        self.conv5 = nn.Conv2d(64, 10, 1, bias=False)

        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.gap(x)
        x = x.view(-1, 10)
        return F.log_softmax(x, dim=1)

model = Net()

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print("Total Parameters:", total_params)
wandb.log({"Total Parameters": total_params})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Total Parameters: 6104


Net(
  (conv1): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (conv2): Sequential(
    (0): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.05, inplace=False)
  )
  (conv3): Sequential(
    (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (conv4): Sequential(
    (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
   

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr=0.01,
momentum=0.9,weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

In [ ]:
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []


In [ ]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()

    correct = 0
    processed = 0
    running_loss = 0.0


    pbar = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)

    for batch_idx, (data, target) in enumerate(pbar):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()


        running_loss += loss.item()

        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        processed += target.size(0)


        pbar.set_postfix(loss=loss.item(), acc=100.0 * correct / processed)


    train_loss = running_loss / len(train_loader)
    train_acc = 100.0 * correct / processed

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    print(f"Epoch {epoch}: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
    wandb.log({"train_loss": train_loss, "train_acc": train_acc})



In [ ]:
def test():
    model.eval()
    test_loss = 0.0
    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)

            output = model(data)
            test_loss += F.nll_loss(output, target, reduction="sum").item()

            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()

    test_loss /= len(test_loader.dataset)
    test_acc = 100 * correct / len(test_loader.dataset)

    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    wandb.log({
        "Test Loss": test_loss,
        "Test Accuracy": test_acc
    })

    print(
        f"Test Loss: {test_loss:.4f} | "
        f"Test Accuracy: {test_acc:.2f}%\n"
    )

In [ ]:
for epoch in range(1, 21):
    train(model, device, train_loader, optimizer,epoch)
    test()
    scheduler.step()

wandb.finish()

Epoch 1: Loss=0.1420, Acc=96.67%
Test Loss: 0.1311 | Test Accuracy: 96.20%



Epoch 2: Loss=0.0970, Acc=97.53%
Test Loss: 0.0610 | Test Accuracy: 98.33%



Epoch 3: Loss=0.0777, Acc=97.94%
Test Loss: 0.0574 | Test Accuracy: 98.29%



Epoch 4: Loss=0.0672, Acc=98.17%
Test Loss: 0.0616 | Test Accuracy: 98.04%



Epoch 5: Loss=0.0594, Acc=98.39%
Test Loss: 0.0425 | Test Accuracy: 98.63%



...

Epoch 17: Loss=0.0331, Acc=99.07%
Test Loss: 0.0218 | Test Accuracy: 99.32%



Epoch 18: Loss=0.0335, Acc=99.06%
Test Loss: 0.0212 | Test Accuracy: 99.34%



Epoch 19: Loss=0.0322, Acc=99.12%
Test Loss: 0.0220 | Test Accuracy: 99.30%



Epoch 20: Loss=0.0334, Acc=99.05%
Test Loss: 0.0202 | Test Accuracy: 99.38%



Test Accuracy,▁▆▆▅▆▇▇▇████████████
Test Loss,█▄▃▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▅▅▆▇▇▇▇▇▇▇▇▇██████
train_loss,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
Test Accuracy,99.38
Test Loss,0.02015
train_acc,99.05167
train_loss,0.03344
